# <span style="color:red"> Lecture 18c: Recoding Variables </span>

<font size = "4">

"Recoding" variables means changing/re-expressing existing variables into a new set of values that may be more useful for data analysis.

Examples:

- A variable called "age" consists of integers beginning at 0 (how old the person is in terms of years). We can transform this into a categorical variable with the values "child", "teenager", "young adult", "adult", "senior"

- A variable called "submission_time" records the time that a student submitted an assignment to Gradescope. We can transform this into a categorical variable with the values "on time" and "late"

- A variable called "candidate_preference" is taken from a survey of people asking which of two presidential candidates they will vote for. This is a categorical variable with the values "Candidate A", "Candidate B", "Don't Know", "Refused", "Blank", "Other Candidate". This is recoded to another categorical variable with only 3 values "Candidate A", "Candidate B", and "Neither".

- We convert a variable "height" measured in inches, to a corresponding variable measuring height in centimeters.

- A variable called "lat" consists of floating point numbers consisting of a geographical coordinate measuring the distance north or south of the Equator. We change this into a categorical variable with the values "north" or "south".

<font size = "4">

We'll perform this last one in **3 different ways** using the circuits data we cleaned last time. In each way, we will consider as "south" those circuits with $-90 < \textrm{lat} \leq 0$ and "north" as circuits with $0 < \textrm{lat} \leq 90$

In [ ]:
import pandas as pd

circuits = pd.read_csv("f1_clean_data/circuits_final.csv")

circuits.head()

,Unnamed: 0,circuitID,circuitRef,name,location,country,lat,lng,alt,url,alt_numeric
0,0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10.0,http://en.wikipedia.org/wiki/Melbourne_Grand_P...,10.0
1,1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,18.0,http://en.wikipedia.org/wiki/Sepang_Internatio...,18.0
2,2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,7.0,http://en.wikipedia.org/wiki/Bahrain_Internati...,7.0
3,3,4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57000,2.26111,109.0,http://en.wikipedia.org/wiki/Circuit_de_Barcel...,109.0
4,4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,130.0,http://en.wikipedia.org/wiki/Istanbul_Park,130.0


<font size = "4">

Here we do it manually with a for loop and an if/else statement:

In [3]:
nrows, ncols = circuits.shape 

col_no = 5 # column 5 is "lat"
hemisphere = []

for k in range(nrows):
    lat_val = circuits.iloc[k, col_no]
    if lat_val <= 0:
        hemisphere.append("South")
    else:
        hemisphere.append("North")

# could overwrite "lat", but is almost always better to keep the original variable
circuits["hemisphere"] = hemisphere

# Note: I made sure the data in this column didn't need to be cleaned first!!
circuits.head()

TypeError: '<=' not supported between instances of 'str' and 'int'

<font size = "4">

Let's drop the "hemisphere" column to verify that the next approach we try actually works

In [4]:
circuits.drop(columns = "hemisphere",inplace = True)
circuits.head()

KeyError: "['hemisphere'] not found in axis"

<font size = "4">

In the next cell, I *attempt* to use `.apply` to recode the data. 

**What error did I make and how should I fix it?**

In [5]:
def check_hemisphere(lat):
    if lat_val <= 0:
        return "South" 
    else:
        return "North"

circuits["hemisphere"] = circuits["lat"].apply(check_hemisphere)

circuits.head()

TypeError: '<=' not supported between instances of 'str' and 'int'

In [ ]:
# dropping the "hemisphere" column again:
circuits.drop(columns = "hemisphere",inplace = True)
circuits.head()

<font size = "4">

Finally, I use the function `pd.cut` which is used to segment data into "intervals" or "bins".

Remember, we are recoding the latitute variable based on the following criteria:

- If $-90 < \textrm{lat} \leq 0$, the circuit is labeled "south"

- If $0 < \textrm{lat} \leq 90$, the circuit is labeled "north"

In [6]:
lat_bins = [-90, 0, 90]

lat_labels = ["South", "North"]
circuits["hemisphere"] = pd.cut(circuits["lat"], 
                                bins = lat_bins,
                                right = True,
                                labels = lat_labels)

# The "right" command indicates that the right endpoint of each interval
# is *inclusive*, i.e. "less than or equal to"


circuits.head()
                            

,Unnamed: 0,circuitID,circuitRef,name,location,country,lat,lng,alt,url,alt_numeric,hemisphere
0,0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10.0,http://en.wikipedia.org/wiki/Melbourne_Grand_P...,10.0,South
1,1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,18.0,http://en.wikipedia.org/wiki/Sepang_Internatio...,18.0,North
2,2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,7.0,http://en.wikipedia.org/wiki/Bahrain_Internati...,7.0,North
3,3,4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57000,2.26111,109.0,http://en.wikipedia.org/wiki/Circuit_de_Barcel...,109.0,North
4,4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,130.0,http://en.wikipedia.org/wiki/Istanbul_Park,130.0,North


<font size = "4">

Alternatively, we can implement it more robustly by recoding the latitude variable by:

- If $-\infty < \textrm{lat} \leq 0$, the circuit is labeled "south"

- If $0 < \textrm{lat} \leq \infty$, the circuit is labeled "north"

(Important in the unlikely event an F1 racetrack is ever built directly at the South Pole)

In [7]:
import numpy as np

lat_bins = [-np.inf, 0, np.inf]

lat_labels = ["South", "North"]
circuits["hemisphere"] = pd.cut(circuits["lat"], 
                                bins = lat_bins,
                                right = True,
                                labels = lat_labels)


# Note: Since we set lat_bins = [-np.inf, 0, np.inf]
# then intervals are "Less than or equal to 0" and "Above 0"

circuits.head()
                      

,Unnamed: 0,circuitID,circuitRef,name,location,country,lat,lng,alt,url,alt_numeric,hemisphere
0,0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10.0,http://en.wikipedia.org/wiki/Melbourne_Grand_P...,10.0,South
1,1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,18.0,http://en.wikipedia.org/wiki/Sepang_Internatio...,18.0,North
2,2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,7.0,http://en.wikipedia.org/wiki/Bahrain_Internati...,7.0,North
3,3,4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57000,2.26111,109.0,http://en.wikipedia.org/wiki/Circuit_de_Barcel...,109.0,North
4,4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,130.0,http://en.wikipedia.org/wiki/Istanbul_Park,130.0,North


In [8]:
help(pd.cut)

Help on function cut in module pandas:

cut(
    x,
    bins,
    right: bool = True,
    labels=None,
    retbins: bool = False,
    precision: int = 3,
    include_lowest: bool = False,
    duplicates: str = 'raise',
    ordered: bool = True
)
    Bin values into discrete intervals.

    Use `cut` when you need to segment and sort data values into bins. This
    function is also useful for going from a continuous variable to a
    categorical variable. For example, `cut` could convert ages to groups of
    age ranges. Supports binning into an equal number of bins, or a
    pre-specified array of bins.

    Parameters
    ----------
    x : 1d ndarray or Series
        The input array to be binned. Must be 1-dimensional.
    bins : int, sequence of scalars, or IntervalIndex
        The criteria to bin by.

        * int : Defines the number of equal-width bins in the range of `x`. The
          range of `x` is extended by .1% on each side to include the minimum
          and maximum v

<font size = "4">

**Exercise:** Recode the altitude variable into a categorial variable with the values

- "Low altitude" if $\textrm{altitude} \leq 45$
- "Medium altitude" if $45 < \textrm{altitude} \leq 145$
- "High altitude" if $145 < \textrm{altitude}$

Try to write the code without guessing/looking at the minimum and maximum value of altitude in the dataset

In [ ]:
# your code here



<font size = "4">

**Exercise:** What happens if you use the ``Pandas`` function ``.cut`` if the bins you provide don't capture all the data in the column? What happens if there are missing values in the column? Test it out by recoding the altitude variable into a categorical variable with the values:

- "Group A" if $0 < \textrm{altitude} \leq 120$
- "Group B" if $120 < \textrm{altitude} \leq 400$
- "Group C" if $400 < \textrm{altitude} \leq 800$

In [ ]:
print(circuits.loc[74, "alt"])
print("When the altitude value is missing, pd.cut assigns:",
      circuits.loc[74, "alt_group"])
print()
print(circuits.loc[70, "alt"])
print("When the altitude is below sea level (negative), pd.cut assigns:",
      circuits.loc[70, "alt_group"])
print()
print(circuits.loc[28, "alt"])
print("When the altitude is higher than 800, pd.cut assigns:",
      circuits.loc[28, "alt_group"])

# <span style="color:red"> Practice: Recoding the 2020 Election Data </span>

<font size = "4">

In these exercises, you'll practice the recoding methods from this lecture on the 2020 U.S. general election data in the folder `election_data`.

Run the cell below to read in the files we'll use.

In [ ]:
house_cand = pd.read_csv("election_data/house_candidate.csv")
house_state = pd.read_csv("election_data/house_state.csv")
pres_county = pd.read_csv("election_data/president_county.csv")
pres_state = pd.read_csv("election_data/president_state.csv")

<font size = "4">

**Exercise 1:** Recoding with a for loop

The `party` column of `house_cand` has many different values. Use `pd.unique()` to see them.

Recode `party` into a new column `party_group` with only three values:

- "Democratic" if party is "DEM"
- "Republican" if party is "REP"
- "Other" for anything else

Use a for loop and an if/elif/else statement, like the first approach in lecture.

In [ ]:
# your answer here

<font size = "4">

**Exercise 2:** Recoding with `.apply`

- Write a function `group_party` that takes a single party abbreviation and returns "Democratic", "Republican", or "Other".

- Use `.apply` to create a column `party_group_2`.

- Check that `party_group` and `party_group_2` agree on every row. (Hint: use `.query()` to look for rows where they are *not* equal.)

- Print the number of candidates in each group.

In [ ]:
# your answer here

<font size = "4">

**Exercise 3:** Recoding with `pd.cut`

Recode the `total_votes` column of `pres_county` into a categorical variable `county_size`:

- "Small" if $\textrm{total votes} \leq 10{,}000$
- "Medium" if $10{,}000 < \textrm{total votes} \leq 100{,}000$
- "Large" if $100{,}000 < \textrm{total votes}$

- First use `bins = [0, 10000, 100000, np.inf]`. Then use `.query("county_size.isna()")` to display the rows where `county_size` is missing, including their `total_votes`.

- Change the first bin edge to `-np.inf`, rerun `pd.cut`, and print the number of rows where `county_size` is missing (it should be 0).

- Print the number of counties in each category.

In [ ]:
# your answer here

In [ ]:
# your answer here

<font size = "4">

**Exercise 4:** Recoding with a dictionary

The U.S. Census Bureau groups the states into four *regions* and nine *divisions*. The table is reproduced on the Wikipedia page [List of regions of the United States](https://en.wikipedia.org/wiki/List_of_regions_of_the_United_States).

The cell below stores the regions in a *dictionary*: each state's name is a *key*, and its region is the corresponding *value*. You can look up a value by putting its key in square brackets, for example `census_region["Georgia"]`.

- Run the cell. Then print the region for Georgia, and for one other state of your choice.

- Print `"Georgia" in census_region` and `"United States" in census_region`.

In [ ]:
census_region = {
    # Northeast
    "Connecticut": "Northeast", "Maine": "Northeast", "Massachusetts": "Northeast",
    "New Hampshire": "Northeast", "Rhode Island": "Northeast", "Vermont": "Northeast",
    "New Jersey": "Northeast", "New York": "Northeast", "Pennsylvania": "Northeast",
    # Midwest
    "Illinois": "Midwest", "Indiana": "Midwest", "Michigan": "Midwest", "Ohio": "Midwest",
    "Wisconsin": "Midwest", "Iowa": "Midwest", "Kansas": "Midwest", "Minnesota": "Midwest",
    "Missouri": "Midwest", "Nebraska": "Midwest", "North Dakota": "Midwest", "South Dakota": "Midwest",
    # South
    "Delaware": "South", "District of Columbia": "South", "Florida": "South", "Georgia": "South",
    "Maryland": "South", "North Carolina": "South", "South Carolina": "South", "Virginia": "South",
    "West Virginia": "South", "Alabama": "South", "Kentucky": "South", "Mississippi": "South",
    "Tennessee": "South", "Arkansas": "South", "Louisiana": "South", "Oklahoma": "South",
    "Texas": "South",
    # West
    "Arizona": "West", "Colorado": "West", "Idaho": "West", "Montana": "West", "Nevada": "West",
    "New Mexico": "West", "Utah": "West", "Wyoming": "West", "Alaska": "West", "California": "West",
    "Hawaii": "West", "Oregon": "West", "Washington": "West"
}

In [ ]:
# your answer here

<font size = "4">

- Write a function `get_region` that takes a state's name and returns its region. If the name is not a key of the dictionary, return "Not a state". (Hint: use `in`, as in the previous cell.)

- Use `.apply` to create a column `region` in `pres_state`.

- Print the number of rows of `pres_state` in each region, including "Not a state".

- `.replace()` also accepts a dictionary. Create a column `region_2` using `pres_state["state"].replace(census_region)`. Then use `.query()` to display the `state`, `region`, and `region_2` columns for the row where `state` is `"United States"`.

In [ ]:
# your answer here

In [ ]:
# your answer here

<font size = "4">

**Exercise 5:** Putting it together: House race margins

- Use `.query()` to keep only the winners in `house_cand` (`won == True`). Merge the `current_votes` column from `house_state` onto the winners (Lecture 18a), and compute each winner's percent of the vote in a column called `win_pct`.

- Recode `win_pct` into a new column `result_type`:

    - "Close" if $\textrm{win pct} \leq 55$
    - "Comfortable" if $55 < \textrm{win pct} \leq 65$
    - "Landslide" if $65 < \textrm{win pct}$

- Print the number of races in each category.

- Use `.query("result_type.isna()")` to display the row with a missing `result_type`, including its `current_votes` and `win_pct`.

In [ ]:
# your answer here

<font size = "4">

**Exercise 6 (Challenge):** Census divisions

The cell below stores the nine Census divisions in a second dictionary, `census_division`.

- Add a `division` column to `pres_county`. (It has one row per county, so each state's name appears many times.)

- Use `.query()` to print the number of rows of `pres_county` in the "New England" division and in the "East South Central" division.

- Display the first 10 New England rows of `pres_county`.

In [ ]:
census_division = {
    "Connecticut": "New England", "Maine": "New England", "Massachusetts": "New England",
    "New Hampshire": "New England", "Rhode Island": "New England", "Vermont": "New England",
    "New Jersey": "Mid-Atlantic", "New York": "Mid-Atlantic", "Pennsylvania": "Mid-Atlantic",
    "Illinois": "East North Central", "Indiana": "East North Central", "Michigan": "East North Central",
    "Ohio": "East North Central", "Wisconsin": "East North Central",
    "Iowa": "West North Central", "Kansas": "West North Central", "Minnesota": "West North Central",
    "Missouri": "West North Central", "Nebraska": "West North Central", "North Dakota": "West North Central",
    "South Dakota": "West North Central",
    "Delaware": "South Atlantic", "District of Columbia": "South Atlantic", "Florida": "South Atlantic",
    "Georgia": "South Atlantic", "Maryland": "South Atlantic", "North Carolina": "South Atlantic",
    "South Carolina": "South Atlantic", "Virginia": "South Atlantic", "West Virginia": "South Atlantic",
    "Alabama": "East South Central", "Kentucky": "East South Central", "Mississippi": "East South Central",
    "Tennessee": "East South Central",
    "Arkansas": "West South Central", "Louisiana": "West South Central", "Oklahoma": "West South Central",
    "Texas": "West South Central",
    "Arizona": "Mountain", "Colorado": "Mountain", "Idaho": "Mountain", "Montana": "Mountain",
    "Nevada": "Mountain", "New Mexico": "Mountain", "Utah": "Mountain", "Wyoming": "Mountain",
    "Alaska": "Pacific", "California": "Pacific", "Hawaii": "Pacific", "Oregon": "Pacific",
    "Washington": "Pacific"
}

In [ ]:
# your answer here